##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Extract music chords from audio with the Gemini API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Audio_chord_extraction.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>

This notebook shows you how to use the Gemini API's audio understanding capabilities to extract musical chord progressions from music.

You will learn how to:

1. Extract chords from a **YouTube music video** with a simple prompt.
2. Use **structured output** with a typed schema for predictable chord data.
3. **Upload your own audio file** and extract chords from it.
4. Display the extracted chord information.

## Setup

### Install the SDK

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata
from google import genai

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

Select the model to use:

In [ ]:
MODEL_ID = "gemini-2.5-flash" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash"] {"allow-input": true}

## Extract chords from a YouTube video

The simplest way to extract chords is to pass a YouTube URL directly. Gemini can analyze the audio track and identify the chord progression. Use JSON mode to get structured output that you can easily process.

In [ ]:
import json
from google.genai import types

youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # @param {type:"string"}

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(
            file_uri=youtube_url,
            mime_type="video/*",
        ),
        """
            Analyze the music in this video and extract the chord
            progression. Return a JSON object with:
            - "title": the song title if identifiable, otherwise "Unknown"
            - "tempo": estimated tempo in BPM
            - "key": the musical key
            - "progression": the chord progression as a string
              (e.g. "Am - G - F - E")
            - "chords": a list of objects, each with "root_note",
              "chord_type", and "chord_notation"
        """,
    ],
    config={
        "response_mime_type": "application/json",
    },
)

chord_data = json.loads(response.text)
print(json.dumps(chord_data, indent=2))

{
  "title": "Never Gonna Give You Up",
  "tempo": 113,
  "key": "A Major",
  "progression": "F#m - D - E - A - F#m - D - E - A - F#m - D - E - A - D - E - F#m - A - D - E - A - D - E - F#m - A",
  "chords": [
    {
      "root_note": "F#",
      "chord_type": "minor",
      "chord_notation": "F#m"
    },
    {
      "root_note": "D",
      "chord_type": "major",
      "chord_notation": "D"
    },
    {
      "root_note": "E",
      "chord_type": "major",
      "chord_notation": "E"
    },
    {
      "root_note": "A",
      "chord_type": "major",
      "chord_notation": "A"
    },
    {
      "root_note": "F#",
      "chord_type": "minor",
      "chord_notation": "F#m"
    },
    {
      "root_note": "D",
      "chord_type": "major",
      "chord_notation": "D"
    },
    {
      "root_note": "E",
      "chord_type": "major",
      "chord_notation": "E"
    },
    {
      "root_note": "A",
      "chord_type": "major",
      "chord_notation": "A"
    },
    {
      "root_note": "F#",
 

## Use structured output with a schema

For more predictable results, define a schema using Python dataclasses and pass it to `response_schema`. This ensures the output always matches your expected format. See the [JSON mode quickstart](../quickstarts/JSON_mode.ipynb) for more details.

### Define the chord data model

In [ ]:
import dataclasses
import enum


class ChordType(enum.Enum):
    MAJOR = "major"
    MINOR = "minor"
    SEVENTH = "7"
    MAJOR_SEVENTH = "maj7"
    MINOR_SEVENTH = "m7"
    SUSPENDED_FOURTH = "sus4"
    SUSPENDED_SECOND = "sus2"
    DIMINISHED = "dim"
    AUGMENTED = "aug"


@dataclasses.dataclass
class Chord:
    root_note: str
    chord_type: ChordType
    chord_notation: str


@dataclasses.dataclass
class ChordProgression:
    title: str
    tempo: str
    key: str
    progression: str
    chords: list[Chord]

### Extract chords with the schema

Now pass the `ChordProgression` dataclass as the `response_schema`. Gemini will return data that matches the schema exactly.

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        types.Part.from_uri(
            file_uri=youtube_url,
            mime_type="video/*",
        ),
        "Analyze the music in this video and extract the chord progression.",
    ],
    config={
        "response_mime_type": "application/json",
        "response_schema": ChordProgression,
    },
)

result = json.loads(response.text)
print(json.dumps(result, indent=2))

{
  "title": "Never Gonna Give You Up",
  "tempo": "118 BPM",
  "key": "F# minor",
  "progression": "F#m - C#m - D - E",
  "chords": [
    {
      "root_note": "F#",
      "chord_type": "minor",
      "chord_notation": "F#m"
    },
    {
      "root_note": "C#",
      "chord_type": "minor",
      "chord_notation": "C#m"
    },
    {
      "root_note": "D",
      "chord_type": "major",
      "chord_notation": "D"
    },
    {
      "root_note": "E",
      "chord_type": "major",
      "chord_notation": "E"
    }
  ]
}

## Display the results

Print the extracted chord information in a human-readable format.

In [ ]:
print(f"Title: {result.get('title', 'Unknown')}")
print(f"Key: {result.get('key', 'Unknown')}")
print(f"Tempo: {result.get('tempo', 'Unknown')}")
print(f"Progression: {result.get('progression', 'Unknown')}")
print("\nChords:")
for chord in result.get("chords", []):
    notation = chord.get("chord_notation", "?")
    root = chord.get("root_note", "?")
    ctype = chord.get("chord_type", "?")
    print(f"  {notation}  (root: {root}, type: {ctype})")

Title: Never Gonna Give You Up
Key: F# minor
Tempo: 118 BPM
Progression: F#m - C#m - D - E

Chords:
  F#m  (root: F#, type: minor)
  C#m  (root: C#, type: minor)
  D  (root: D, type: major)
  E  (root: E, type: major)

## Use your own audio file

You can also extract chords from audio files that you upload with the [File API](../quickstarts/File_API.ipynb). This is useful when working with local recordings or audio files that are not on YouTube.

Here is an example using a sample music file. Replace the URL with your own audio file, or upload one directly in the Colab file browser.

In [ ]:
AUDIO_URL = "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3"  # @param {type:"string"}
!wget -q $AUDIO_URL -O sample.mp3

Listen to the audio to verify it loaded correctly:

In [ ]:
from IPython.display import Audio

Audio("sample.mp3")

<IPython.lib.display.Audio object>

Upload the file to the Gemini API and extract the chords:

In [ ]:
audio_file = client.files.upload(file="sample.mp3")
print(f"Uploaded: {audio_file.name}")

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        audio_file,
        "Analyze this audio and extract the chord progression.",
    ],
    config={
        "response_mime_type": "application/json",
        "response_schema": ChordProgression,
    },
)

file_result = json.loads(response.text)
print(json.dumps(file_result, indent=2))

Uploaded: files/39kjhovbz57q
{
  "title": "Synth Groove Progression",
  "tempo": "128 BPM",
  "key": "C Minor",
  "progression": "Cm - Gm - Ab - Eb",
  "chords": [
    {
      "root_note": "C",
      "chord_type": "minor",
      "chord_notation": "Cm"
    },
    {
      "root_note": "G",
      "chord_type": "minor",
      "chord_notation": "Gm"
    },
    {
      "root_note": "Ab",
      "chord_type": "major",
      "chord_notation": "Ab"
    },
    {
      "root_note": "Eb",
      "chord_type": "major",
      "chord_notation": "Eb"
    }
  ]
}

## Next steps

### Useful API references:

- Learn more about [audio understanding](https://ai.google.dev/gemini-api/docs/audio) in the Gemini API docs.
- See the [Audio quickstart](../quickstarts/Audio.ipynb) for more audio use cases.
- Check out the [JSON mode quickstart](../quickstarts/JSON_mode.ipynb) for more on structured output.
- Explore the [File API quickstart](../quickstarts/File_API.ipynb) for more on file uploads.

### Related examples:

- [Voice memos](Voice_memos.ipynb): Summarize voice memos with Gemini.